In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime, timedelta

In [2]:
NIFTY500_TICKERS = {
    # Banking & Finance
    "HDFCBANK.NS":   "Banking",
    "ICICIBANK.NS":  "Banking",
    "KOTAKBANK.NS":  "Banking",
    "AXISBANK.NS":   "Banking",
    "SBIN.NS":       "Banking",
    "INDUSINDBK.NS": "Banking",
    "BANDHANBNK.NS": "Banking",
    "FEDERALBNK.NS": "Banking",
    "BAJFINANCE.NS": "NBFC",
    "BAJAJFINSV.NS": "NBFC",
    "CHOLAFIN.NS":   "NBFC",
    "MUTHOOTFIN.NS": "NBFC",
    "HDFCLIFE.NS":   "Insurance",
    "SBILIFE.NS":    "Insurance",
    "ICICIPRULI.NS": "Insurance"
}

In [3]:
INDEX_URLS = {
    "Nifty 50":          "ind_nifty50list.csv",
    "Nifty 500":         "ind_nifty500list.csv",
    "Nifty Midcap 150":  "ind_niftymidcap150list.csv",
    "Nifty Smallcap 250":"ind_niftysmallcap250list.csv",
    "Nifty Midcap 50":   "ind_niftymidcap50list.csv",
    "Nifty Next 50":     "ind_niftynext50list.csv",
    "Nifty Bank":        "ind_niftybanklist.csv",
    "Nifty IT":          "ind_niftyittlist.csv",
}

In [4]:
def download_data(tickers, period="14mo"):
    """
    Download OHLCV data for all tickers.
    Returns dict of {ticker: DataFrame}
    """
    print(f"\n{'═'*60}")
    print(f"  Downloading data for {len(tickers)} stocks...")
    print(f"{'═'*60}")

    data = {}
    failed = []

    # Download in batches of 20 to avoid rate limits
    ticker_list = list(tickers.keys())
    batch_size  = 20

    for i in range(0, len(ticker_list), batch_size):
        batch = ticker_list[i:i+batch_size]
        print(f"  Batch {i//batch_size + 1}: {', '.join([t.replace('.NS','') for t in batch])}")

        try:
            raw = yf.download(
                batch,
                period=period,
                auto_adjust=True,
                progress=False,
                threads=True,
            )

            for ticker in batch:
                try:
                    if len(batch) == 1:
                        df = raw.copy()
                    else:
                        df = raw.xs(ticker, axis=1, level=1).copy()

                    df = df.dropna(subset=["Close"])

                    if len(df) < 200:
                        failed.append(ticker)
                        continue

                    data[ticker] = df

                except Exception:
                    failed.append(ticker)

        except Exception as e:
            print(f"  Batch failed: {e}")
            failed.extend(batch)

        time.sleep(0.5)  # be polite to Yahoo Finance

    print(f"\n  Downloaded: {len(data)} stocks  |  Failed: {len(failed)}")
    if failed:
        print(f"  Failed tickers: {', '.join([t.replace('.NS','') for t in failed[:10]])}")

    return data

In [5]:
data = download_data(NIFTY500_TICKERS)


════════════════════════════════════════════════════════════
════════════════════════════════════════════════════════════
  Batch 1: HDFCBANK, ICICIBANK, KOTAKBANK, AXISBANK, SBIN, INDUSINDBK, BANDHANBNK, FEDERALBNK, BAJFINANCE, BAJAJFINSV, CHOLAFIN, MUTHOOTFIN, HDFCLIFE, SBILIFE, ICICIPRULI

  Downloaded: 15 stocks  |  Failed: 0


In [6]:
data['HDFCBANK.NS']

Price,Close,High,Low,Open,Volume
Date,,,,,
2025-03-24,887.996948,890.167619,874.997672,877.636984,17393736
2025-03-25,898.578918,909.555548,888.736976,890.414316,39101416
2025-03-26,891.228333,901.563607,888.983649,900.330278,24478442
2025-03-27,900.502930,908.223582,887.059642,887.996984,41658100
2025-03-28,901.908936,905.756947,890.882985,902.303625,28773648
...,...,...,...,...,...
2026-05-18,768.650024,774.099976,751.349976,759.000000,29812079
2026-05-19,762.450012,770.799988,760.250000,766.700012,40307830
2026-05-20,759.500000,762.250000,755.150024,759.599976,24013451


In [7]:
INDEX_URLS = {
    "Nifty 50":          "ind_nifty50list.csv",
    "Nifty 500":         "ind_nifty500list.csv",
    "Nifty Midcap 150":  "ind_niftymidcap150list.csv",
    "Nifty Smallcap 250":"ind_niftysmallcap250list.csv",
    "Nifty Midcap 50":   "ind_niftymidcap50list.csv",
    "Nifty Next 50":     "ind_niftynext50list.csv",
    "Nifty Bank":        "ind_niftybanklist.csv",
    "Nifty IT":          "ind_niftyittlist.csv",
}

BASE = "https://archives.nseindia.com/content/indices/"

def get_index_tickers(index_name):
    url = BASE + INDEX_URLS[index_name]
    df  = pd.read_csv(url)
    return [s + ".NS" for s in df['Symbol'].tolist()]

# Example
tickers = get_index_tickers("Nifty Smallcap 250")
print(f"Got {len(tickers)} tickers")

Got 250 tickers


In [8]:
tickers

['ACMESOLAR.NS',
 'AADHARHFC.NS',
 'AARTIIND.NS',
 'AAVAS.NS',
 'ACE.NS',
 'ACUTAAS.NS',
 'ABFRL.NS',
 'ABLBL.NS',
 'ABREL.NS',
 'ABSLAMC.NS',
 'CPPLUS.NS',
 'AEGISLOG.NS',
 'AEGISVOPAK.NS',
 'AFCONS.NS',
 'AFFLE.NS',
 'ABDL.NS',
 'ARE&M.NS',
 'AMBER.NS',
 'ANANDRATHI.NS',
 'ANANTRAJ.NS',
 'ANGELONE.NS',
 'ANURAS.NS',
 'APTUS.NS',
 'ASAHIINDIA.NS',
 'ASTERDM.NS',
 'ATHERENERG.NS',
 'ATUL.NS',
 'BEML.NS',
 'BLS.NS',
 'BALRAMCHIN.NS',
 'BANDHANBNK.NS',
 'BATAINDIA.NS',
 'BAYERCROP.NS',
 'BELRISE.NS',
 'BIKAJI.NS',
 'BSOFT.NS',
 'BLUEDART.NS',
 'BLUEJET.NS',
 'BBTC.NS',
 'FIRSTCRY.NS',
 'BRIGADE.NS',
 'MAPMYINDIA.NS',
 'CCL.NS',
 'CESC.NS',
 'CIEINDIA.NS',
 'CANFINHOME.NS',
 'CANHLIFE.NS',
 'CAPLIPOINT.NS',
 'CGCL.NS',
 'CARBORUNIV.NS',
 'CARTRADE.NS',
 'CASTROLIND.NS',
 'CEATLTD.NS',
 'CEMPRO.NS',
 'CENTRALBK.NS',
 'CDSL.NS',
 'CHALET.NS',
 'CHAMBLFERT.NS',
 'CHENNPETRO.NS',
 'CHOICEIN.NS',
 'CHOLAHLDNG.NS',
 'CUB.NS',
 'CLEAN.NS',
 'COHANCE.NS',
 'CAMS.NS',
 'CONCORDBIO.NS',
 'CRAFTSMAN

In [9]:
stock = yf.Ticker("RELIANCE.NS")

info = stock.info 

In [10]:
info

{'address1': 'Maker Chambers IV',
 'address2': '3rd Floor 222 Nariman Point',
 'city': 'Mumbai',
 'zip': '400021',
 'country': 'India',
 'phone': '91 22 3555 5000',
 'fax': '91 22 2204 2268',
 'website': 'https://www.ril.com',
 'industry': 'Oil & Gas Refining & Marketing',
 'industryKey': 'oil-gas-refining-marketing',
 'industryDisp': 'Oil & Gas Refining & Marketing',
 'sector': 'Energy',
 'sectorKey': 'energy',
 'sectorDisp': 'Energy',
 'longBusinessSummary': 'Reliance Industries Limited engages in the hydrocarbon exploration and production, oil and chemicals, retail, and digital service businesses worldwide. It operates through Oil to Chemicals, Oil and Gas, Retail, Digital Services, and Others segments. The company offers refining and marketing products, including liquefied petroleum gas, propylene, naphtha, gasoline, jet/aviation turbine fuel, kerosine oil, diesel, sulphur, and petroleum coke. It also provides polymers, including high-density and low-density polyethylene (PE), line